# IndicDocLayout — data and training

Training the **layout** model: detection and reading order, learned jointly in one pass.

> The recognizer (IndicBlockOCR) is **not** trained here. It ships as a released
> checkpoint and this repo carries no recipe for it.

The pipeline is three steps, each of which writes a file the next one reads:

```
configs/ocr/data/sources.yaml --splits--> layout_{train,val,test}.json
layout_train.json             --pack----> <cache>_shard*.blob + _meta.npz
cache + configs/ocr/train/    --train---> runs/layout/final (+ /ema)
```

In [ ]:
from importlib.metadata import PackageNotFoundError, version

import torch

import bodhan_genai.ocr

print("bodhan-genai:", bodhan_genai.ocr.__version__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
for package in ("transformers", "accelerate", "albumentations"):
    try:
        print(f"{package}:", version(package))
    except PackageNotFoundError:
        print(f"{package}: not installed")

## The taxonomy

37 classes. **The ids are baked into every checkpoint** — reordering the list silently
relabels every model ever trained, so the list is append-only.

Note this is the *training* taxonomy, and a different thing from the inference contract:
the contract maps these raw labels onto the handful of coarse types that pick a prompt.

In [ ]:
from bodhan_genai.ocr.data import CLASSES, NUM_CLASSES

print(NUM_CLASSES, "classes")
print(" 0-24 education   :", ", ".join(CLASSES[:25]))
print("25-26 printed-only:", ", ".join(CLASSES[25:27]))
print("27-36 furniture   :", ", ".join(CLASSES[27:]))

### The page parser

`labels_from_doc` is the one parser used by the packer, the trainer and the eval, so the
splits cannot disagree about what an annotation means.

Two things it does that are easy to miss: bboxes are **y first** in thousandths, and the
optional `metadata.header` / `metadata.footer` regions are appended with ranks either
side of the content so the header reads first and the footer last.

In [ ]:
from bodhan_genai.ocr.data import labels_from_doc

doc = {
    "content": [
        {"bbox": [100, 50, 200, 900], "label": "Title", "reading_order": 1},
        {"bbox": [250, 50, 600, 900], "label": "Paragraph", "reading_order": 2},
        {"bbox": [0, 0, 0, 0], "label": "Paragraph", "reading_order": 3},  # degenerate
        {"bbox": [700, 50, 800, 900], "label": "Nonsense", "reading_order": 4},  # unknown
    ],
    "metadata": {"footer": {"bbox": [950, 100, 980, 900]}},
}
boxes, classes, order = labels_from_doc(doc)
print("kept", len(boxes), "boxes; classes", classes, "; order", order)
print("(the degenerate box and the unknown label are dropped; the footer reads last)")

## 1. Splits

Sources do not all admit the same policy, so each declares its own:

| policy | when |
| --- | --- |
| `native` | the source ships `manifest_by_split.json` — use it, or its numbers stop being comparable |
| `holdout` | a val/test set was fixed before this pipeline and models were measured on it |
| `hash` | fresh deterministic 80/10/10 on `md5(stem)` |

`md5` and not `hash()`: the builtin is salted per process, so a rebuild would reshuffle
the split and leak test pages into train.

In [ ]:
from collections import Counter

from bodhan_genai.ocr.data import load_sources
from bodhan_genai.ocr.data.splits import hash_bucket

sources, data_root = load_sources("../../configs/ocr/data/sources.yaml")
print("data_root:", data_root)
for spec in sources:
    print(f"  {spec.name:<22} {spec.domain:<12} policy={spec.policy:<8} weight={spec.weight}")

print()
print("hash policy over 20k stems:", Counter(hash_bucket(f"s{i}") for i in range(20_000)))

In [ ]:
# Build the manifests (reads only jsons/, so it is cheap and safe to re-run):
#
#   scripts/ocr/splits.sh
#
# from python:
# from bodhan_genai.ocr.data import build_manifests
# from bodhan_genai.ocr.data.splits import write_manifests
# write_manifests(build_manifests(sources, data_root), "data/manifests")

### Count what is actually in the mix

37 classes with several partially-annotated sources means some classes may have almost
no examples. A class with forty instances trains to noise, and the only way to find that
out before a multi-day run is to count.

In [ ]:
# from bodhan_genai.ocr.data import summarize
#
# summary = summarize("data/manifests/layout_train.json")
# print(summary.pages, "pages,", summary.blocks, "blocks")
# print("absent classes:", summary.absent_labels)

## 2. Pack the cache

Training reads every page every epoch. As loose files that is two filesystem opens per
page per epoch, which on a shared parallel filesystem costs more than the forward pass.
So pages are packed once into a few big blobs plus one metadata array.

    scripts/ocr/pack.sh

Pages that fail to open, or that parse to zero boxes, are skipped with a warning —
`PackStats.skipped` is how you notice a pack that quietly dropped a tenth of the corpus.

In [ ]:
# from bodhan_genai.ocr.data import BlobCache, pack
#
# stats = pack("data/manifests/layout_train.json", "data/cache/layout_train")
# print(stats)
#
# with BlobCache("data/cache/layout_train") as cache:
#     print(len(cache), "pages;", cache.source_names)
#     boxes, classes, order = cache.labels(0)
#     print("page 0:", len(boxes), "boxes")

## The reading-order objective

The detector predicts an antisymmetric matrix `S` over its queries: `S[i, j] > 0` means
query *i* is read before *j*. Two choices worth understanding before changing anything:

* **GCE, not BCE** — reading-order annotation is genuinely ambiguous on multi-column
  pages, and `q` bounds what one confidently-wrong pair can contribute.
* **Locality weighting** — pairs are weighted `exp(-|Δrank| / tau)`, so the loss spends
  its capacity where a human would notice the mistake.

Only the strict upper triangle is trained: PP-DocLayoutV3's GlobalPointer head masks
`a >= b` to `-1e4`, so the lower triangle holds no real score.

In [ ]:
import torch

from bodhan_genai.ocr.training import decode_order, locality_gce


def scores_for(order):
    order = torch.as_tensor(order, dtype=torch.float32)
    return (order.unsqueeze(1) - order.unsqueeze(0)) * -5.0


order = torch.arange(8, dtype=torch.float32)
correct = scores_for(order)

adjacent = correct.clone()
adjacent[3, 4], adjacent[4, 3] = -correct[3, 4], -correct[4, 3]
distant = correct.clone()
distant[0, 7], distant[7, 0] = -correct[0, 7], -correct[7, 0]

print(f"correct              {locality_gce(correct, order):.4f}")
print(f"one adjacent pair    {locality_gce(adjacent, order):.4f}   <- costs more")
print(f"one distant pair     {locality_gce(distant, order):.4f}")
print(f"fully reversed       {locality_gce(-correct, order):.4f}")
print()
print("decode_order recovers a permutation:", decode_order(scores_for([3, 0, 4, 1, 2])).tolist())

## The batch mix

`MixedSourceSampler` composes **every** batch to the configured ratio rather than
shuffling a weighted pool. A weighted pool is right in expectation over an epoch, but any
individual step can be almost all one source — and with gradient accumulation across DDP
ranks, that is what the optimizer actually sees.

In [ ]:
from collections import Counter

import numpy as np

from bodhan_genai.ocr.training import MixedSourceSampler

source_ids = np.concatenate([np.full(600, 0), np.full(300, 1), np.full(100, 2)])
sampler = MixedSourceSampler(source_ids, {0: 0.6, 1: 0.3, 2: 0.1}, batch_size=10, seed=1)

print("per-batch counts:", sampler.counts)
for batch in list(sampler)[:3]:
    print("  batch mix:", dict(Counter(int(source_ids[i]) for i in batch)))

## 3. Train

Every key in `configs/ocr/train/layout.yaml` maps to a field of `LayoutTrainConfig`, and
an **unknown key raises** rather than falling back to a default — a typo'd
`learing_rate` is otherwise something you discover from the loss curve three days in.

Two settings that are load-bearing:

* `backbone_learning_rate` is an order of magnitude below the head rate. The backbone
  arrives document-pretrained; driving it at the head's rate erases that in a few hundred
  steps and the run never recovers.
* `max_grad_norm: 0.1` is deliberately tight. Hungarian matching makes the loss
  discontinuous, so a step that flips an assignment produces a very large gradient.

In [ ]:
from bodhan_genai.ocr.training import LayoutTrainConfig

config = LayoutTrainConfig.from_yaml("../../configs/ocr/train/layout.yaml")
for key in (
    "checkpoint",
    "lambda_order",
    "epochs",
    "batch_size",
    "learning_rate",
    "backbone_learning_rate",
    "max_grad_norm",
    "use_ema",
):
    print(f"  {key:<24} {getattr(config, key)}")

In [ ]:
# Smoke-test the recipe on a handful of batches before committing a multi-day run:
#
#   scripts/ocr/train.sh --max-steps 20
#
# Full run, two GPUs:
#
#   GPUS=0,1 scripts/ocr/train.sh
#
# Both the live weights and the EMA shadow are written at every checkpoint — EMA is
# usually the better model on this corpus, but not always, and re-running is expensive.

## 4. Evaluate

    scripts/ocr/eval.sh --ckpt runs/layout/final

Read `tau_model_hard`, not just the aggregate. The raster baseline — top-to-bottom,
left-to-right — is very strong on single-column pages, so a model that has learned
nothing about ordering still scores well overall. The hard slice is the pages where
raster is *wrong*, and that is where the order head either earns its place or does not.

In [ ]:
from bodhan_genai.ocr.eval import DetectionAccumulator, kendall_tau, raster_order

# the metrics are plain functions and can be exercised without a checkpoint
gt_boxes, gt_classes = [[0, 0, 1, 1], [2, 2, 3, 3]], [0, 1]
accumulator = DetectionAccumulator()
accumulator.add(gt_boxes, gt_classes, [0.9, 0.8], gt_boxes, gt_classes)
print("a perfect detector:", accumulator.summary()["mAP50"])

two_columns = [
    [0.25, 0.2, 0.4, 0.1],
    [0.75, 0.2, 0.4, 0.1],
    [0.25, 0.8, 0.4, 0.1],
    [0.75, 0.8, 0.4, 0.1],
]
raster = raster_order(two_columns)
true_order = [0, 2, 1, 3]  # a real two-column page reads down each column
print(
    "raster ranks:",
    raster,
    "| tau vs true column order:",
    round(kendall_tau(raster, true_order), 3),
)
print("(raster is confidently wrong here — this is what the hard slice measures)")

## Next

* `docs/ocr/end-to-end.md` — the whole pipeline on one page
* `configs/ocr/data/sources.yaml` — the corpus and the annotation schema
* `notebooks/ocr/inference.ipynb` — running the trained model